# Day 3 — Single-Asset Barrier Conditioning

## tl;dr

For the same monthly monitored down-and-out call, the four estimators agree within
replication uncertainty: the largest absolute paired direct-versus-conditioned
\(z\)-score is **1.43**.

The variance-reduction result is strongly barrier-dependent:

- at a 70% barrier, conditioning is essentially neutral (MC VRF **1.00×**, RQMC VRF **1.02×**);
- at 85%, conditioning VRF rises to **1.09×** for MC and **2.72×** for RQMC;
- at 95%, conditioning VRF reaches **1.67×** for MC and **22.53×** for RQMC.

RQMC benefits much more once the barrier indicator is smoothed. At the 95% barrier,
the MC-to-RQMC variance gain is **20.29×** for the direct estimator and **273.44×**
after conditioning, an interaction ratio of **13.48×**.

The Greek diagnostics tell the same story. In the 95% case, conditioned-RQMC Delta
replication SD is about **0.00076** across all four spot bumps, versus **0.043** for
direct RQMC at the 0.1% bump. Conditioned-RQMC Vega SD is about **0.00035** across
all three vol bumps, whereas direct RQMC ranges from **0.0176** to **0.0088**.
Direct Gamma is genuinely unstable: at the 0.1% spot bump its replication SD is
**0.893** for MC and **1.113** for RQMC. With conditioning, Gamma stays near
**0.0144** and its replication SD falls below **0.0008** for MC and **0.00034** for
RQMC.


## Context & Methods

The bridge product is a one-year, monthly monitored down-and-out call with \(S_0=K=100\).
The SPX dividend yield and 12-month ATM volatility, plus the latest positive 3Y SOFR proxy,
are read from `Bloomberg_Real_Autocallable_Market_Data_HSBC.xlsx`.

Under risk-neutral GBM,

\[
\log S_{t+\Delta t}=\log S_t+
(r-q-\tfrac12\sigma^2)\Delta t+\sigma\sqrt{\Delta t}Z.
\]

For the conditioned estimator, survival over the next monitoring date requires
\(Z>\ell\), where

\[
\ell=\frac{\log(H/S_t)-(r-q-\tfrac12\sigma^2)\Delta t}
{\sigma\sqrt{\Delta t}}.
\]

With \(p=1-\Phi(\ell)\), the one-step survival transform is

\[
Z=\Phi^{-1}\left(\Phi(\ell)+pU\right),
\qquad
W\leftarrow Wp.
\]

The final conditioned payoff is

\[
e^{-rT}(S_T-K)^+W.
\]

### Key Assumptions

- Barrier monitoring is **monthly and discrete**; the continuous-barrier analytic price is a diagnostic only.
- Contract strike and absolute barrier stay fixed under spot bumps.
- Direct and conditioned estimators within a sampling family use the same uniforms.
- All spot/volatility bump scenarios within a replication reuse the same uniforms (CRN).
- RQMC uses independent Owen-scrambled Sobol replications and \(N=2^m\).
- Uniforms are clipped before inverse-normal transformation.
- Vega is reported per one volatility point; Gamma is per squared spot unit.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import tempfile
import time

import numpy as np
import pandas as pd
import openpyxl
from scipy.stats import norm, qmc

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "hsbc_day3_mpl_cache")
)
import matplotlib.pyplot as plt

pd.set_option("display.precision", 8)
plt.style.use("seaborn-v0_8-whitegrid")


def find_project_root():
    candidates = []
    override = os.environ.get("AP_PROJECT_ROOT")
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    notebook_dir = Path(globals().get("__file__", Path.cwd())).resolve().parent
    candidates.extend([notebook_dir, *notebook_dir.parents])
    for candidate in candidates:
        if (candidate / "config" / "core_project_config.json").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate config/core_project_config.json. "
        "Run this notebook from the project directory or set AP_PROJECT_ROOT."
    )


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


PROJECT_DIR = find_project_root()
CONFIG_FILE = PROJECT_DIR / "config" / "core_project_config.json"
with CONFIG_FILE.open(encoding="utf-8") as stream:
    PROJECT_CONFIG = json.load(stream)

SOURCE_FILE = PROJECT_DIR / PROJECT_CONFIG["market_data"]["relative_path"]
assert SOURCE_FILE.is_file(), f"Source workbook not found: {SOURCE_FILE}"
SOURCE_SHA256 = sha256_file(SOURCE_FILE)
EXPECTED_SHA256 = PROJECT_CONFIG["market_data"]["sha256"].upper()
assert SOURCE_SHA256 == EXPECTED_SHA256, (
    f"Workbook hash mismatch: expected {EXPECTED_SHA256}, got {SOURCE_SHA256}"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "day3_barrier_conditioning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_SEED = 20260729
S0 = 100.0
STRIKE = 100.0
T = 1.0
N_STEPS = 12
BARRIER_RATIOS = np.array([0.70, 0.85, 0.95])
SPOT_BUMPS = np.array([0.001, 0.0025, 0.005, 0.01])
VOL_BUMPS = np.array([0.0025, 0.0050, 0.0100])
N_MAIN = 2**14
R_MAIN = 24
N_REFERENCE = 2**18
R_REFERENCE = 16
UNIFORM_EPS = np.finfo(float).eps

METHOD_ORDER = [
    "Direct MC",
    "Direct RQMC",
    "Conditioned MC",
    "Conditioned RQMC",
]
METHOD_COLORS = {
    "Direct MC": "#4C78A8",
    "Direct RQMC": "#F58518",
    "Conditioned MC": "#54A24B",
    "Conditioned RQMC": "#E45756",
}

print(f"Project root: {PROJECT_DIR}")
print(f"Source: {SOURCE_FILE}")
print(f"Verified SHA-256: {SOURCE_SHA256}")
print(
    f"Main experiment: N={N_MAIN:,}, replications={R_MAIN}; "
    f"reference: N={N_REFERENCE:,}, scrambles={R_REFERENCE}"
)

## Data

### 1. Read and validate the SPX market inputs

In [ ]:
def is_positive_number(value):
    return (
        isinstance(value, (int, float, np.integer, np.floating))
        and not isinstance(value, bool)
        and np.isfinite(value)
        and value > 0
    )


def read_single_asset_inputs(path):
    workbook = openpyxl.load_workbook(path, data_only=True, read_only=False)
    snapshot = workbook["Underlying_Snapshot"]
    market_history = workbook["Market_History"]

    ticker = snapshot["B6"].value
    dividend_yield_pct = snapshot["E6"].value
    volatility_pct = snapshot["H6"].value

    if not is_positive_number(volatility_pct):
        raise ValueError("SPX 12M ATM volatility is missing or non-positive.")
    if not is_positive_number(dividend_yield_pct):
        raise ValueError("SPX dividend yield is missing or non-positive.")

    rate_observations = []
    for row in range(6, market_history.max_row + 1):
        date_value = market_history.cell(row, 10).value
        rate_pct = market_history.cell(row, 11).value
        if hasattr(date_value, "year") and is_positive_number(rate_pct):
            rate_observations.append(
                (pd.Timestamp(date_value), float(rate_pct) / 100.0)
            )
    if not rate_observations:
        raise ValueError("No positive 3Y SOFR proxy observations found.")

    rate_series = pd.Series(
        dict(rate_observations), name="3Y SOFR proxy"
    ).sort_index()
    return {
        "ticker": ticker,
        "q": float(dividend_yield_pct) / 100.0,
        "sigma": float(volatility_pct) / 100.0,
        "r": float(rate_series.iloc[-1]),
        "rate_as_of": rate_series.index[-1],
        "snapshot_spot": snapshot["D6"].value,
    }


market = read_single_asset_inputs(SOURCE_FILE)
RISK_FREE_RATE = market["r"]
DIVIDEND_YIELD = market["q"]
BASE_VOLATILITY = market["sigma"]

input_table = pd.DataFrame(
    {
        "Input": [
            "Underlying",
            "Normalised spot",
            "Strike",
            "Risk-free rate",
            "Dividend yield",
            "12M ATM volatility",
            "Monitoring",
        ],
        "Value": [
            market["ticker"],
            S0,
            STRIKE,
            RISK_FREE_RATE,
            DIVIDEND_YIELD,
            BASE_VOLATILITY,
            f"{N_STEPS} monthly dates",
        ],
        "Source / convention": [
            "Underlying_Snapshot",
            "Normalised experiment scale",
            "ATM strike fixed under bumps",
            f"Market_History, {market['rate_as_of'].date()}",
            "Underlying_Snapshot",
            "Underlying_Snapshot",
            "Project bridge experiment",
        ],
    }
)
display(input_table)

assert 0 < DIVIDEND_YIELD < 0.20
assert 0 < BASE_VOLATILITY < 1.00
assert 0 < RISK_FREE_RATE < 0.20

### 2. Build the direct and one-step survival estimators

In [ ]:
def is_power_of_two(value):
    return value > 0 and (value & (value - 1)) == 0


def replication_seed(stage, sampling, replication):
    stage_offsets = {"main": 0, "reference": 100_000_000}
    sampling_offsets = {"MC": 0, "RQMC": 10_000_000}
    return (
        BASE_SEED
        + stage_offsets[stage]
        + sampling_offsets[sampling]
        + replication
    )


def generate_uniforms(sampling, sample_size, dimension, seed):
    if not is_power_of_two(sample_size):
        raise ValueError("Sobol-compatible sample sizes must be powers of two.")
    if sampling == "MC":
        uniforms = np.random.default_rng(seed).random(
            (sample_size, dimension)
        )
    elif sampling == "RQMC":
        engine = qmc.Sobol(d=dimension, scramble=True, seed=seed)
        uniforms = engine.random_base2(m=int(np.log2(sample_size)))
    else:
        raise ValueError("sampling must be 'MC' or 'RQMC'")
    return np.clip(uniforms, UNIFORM_EPS, 1.0 - UNIFORM_EPS)


def direct_payoffs(uniforms, spots, sigmas, barrier):
    spots = np.atleast_1d(np.asarray(spots, dtype=float))
    sigmas = np.atleast_1d(np.asarray(sigmas, dtype=float))
    if spots.shape != sigmas.shape:
        raise ValueError("spots and sigmas must have the same shape")
    if np.any(sigmas <= 0):
        raise ValueError("all volatilities must be positive")

    n_paths, n_steps = uniforms.shape
    n_scenarios = spots.size
    dt = T / n_steps
    sqrt_dt = np.sqrt(dt)
    state = np.broadcast_to(spots, (n_paths, n_scenarios)).copy()
    alive = np.broadcast_to(
        spots > barrier, (n_paths, n_scenarios)
    ).copy()
    normals = norm.ppf(uniforms)

    for step in range(n_steps):
        drift = (
            RISK_FREE_RATE
            - DIVIDEND_YIELD
            - 0.5 * sigmas**2
        ) * dt
        state *= np.exp(
            drift + sigmas * sqrt_dt * normals[:, step, None]
        )
        alive &= state > barrier

    return (
        np.exp(-RISK_FREE_RATE * T)
        * np.maximum(state - STRIKE, 0.0)
        * alive
    )


def conditioned_payoffs(uniforms, spots, sigmas, barrier):
    spots = np.atleast_1d(np.asarray(spots, dtype=float))
    sigmas = np.atleast_1d(np.asarray(sigmas, dtype=float))
    if spots.shape != sigmas.shape:
        raise ValueError("spots and sigmas must have the same shape")
    if np.any(sigmas <= 0):
        raise ValueError("all volatilities must be positive")

    n_paths, n_steps = uniforms.shape
    n_scenarios = spots.size
    dt = T / n_steps
    sqrt_dt = np.sqrt(dt)
    state = np.broadcast_to(spots, (n_paths, n_scenarios)).copy()
    weight = np.broadcast_to(
        (spots > barrier).astype(float), (n_paths, n_scenarios)
    ).copy()
    log_barrier = np.log(barrier)

    for step in range(n_steps):
        drift = (
            RISK_FREE_RATE
            - DIVIDEND_YIELD
            - 0.5 * sigmas**2
        ) * dt
        vol_step = sigmas * sqrt_dt
        lower_bound = (
            log_barrier - np.log(state) - drift
        ) / vol_step
        lower_cdf = norm.cdf(lower_bound)
        survival_probability = np.clip(
            1.0 - lower_cdf, 0.0, 1.0
        )
        transformed_uniform = np.clip(
            lower_cdf
            + survival_probability * uniforms[:, step, None],
            UNIFORM_EPS,
            1.0 - UNIFORM_EPS,
        )
        truncated_normal = norm.ppf(transformed_uniform)
        state *= np.exp(drift + vol_step * truncated_normal)
        weight *= survival_probability

    return (
        np.exp(-RISK_FREE_RATE * T)
        * np.maximum(state - STRIKE, 0.0)
        * weight
    )


def continuous_down_and_out_call(s0, strike, barrier, r, q, sigma, maturity):
    if barrier >= s0:
        return 0.0
    if barrier >= strike:
        raise ValueError("Compact formula requires barrier < strike.")
    vol_t = sigma * np.sqrt(maturity)
    lam = (r - q + 0.5 * sigma**2) / sigma**2
    x1 = np.log(s0 / strike) / vol_t + lam * vol_t
    y1 = (
        np.log(barrier**2 / (s0 * strike)) / vol_t
        + lam * vol_t
    )
    vanilla = (
        s0 * np.exp(-q * maturity) * norm.cdf(x1)
        - strike * np.exp(-r * maturity) * norm.cdf(x1 - vol_t)
    )
    image = (
        s0
        * np.exp(-q * maturity)
        * (barrier / s0) ** (2 * lam)
        * norm.cdf(y1)
        - strike
        * np.exp(-r * maturity)
        * (barrier / s0) ** (2 * lam - 2)
        * norm.cdf(y1 - vol_t)
    )
    return vanilla - image


scenario_rows = [{"Scenario": "base", "Spot": S0, "Sigma": BASE_VOLATILITY}]
for bump in SPOT_BUMPS:
    scenario_rows.extend(
        [
            {
                "Scenario": f"spot_down_{bump:.4f}",
                "Spot": S0 * (1.0 - bump),
                "Sigma": BASE_VOLATILITY,
            },
            {
                "Scenario": f"spot_up_{bump:.4f}",
                "Spot": S0 * (1.0 + bump),
                "Sigma": BASE_VOLATILITY,
            },
        ]
    )
for bump in VOL_BUMPS:
    scenario_rows.extend(
        [
            {
                "Scenario": f"vol_down_{bump:.4f}",
                "Spot": S0,
                "Sigma": BASE_VOLATILITY - bump,
            },
            {
                "Scenario": f"vol_up_{bump:.4f}",
                "Spot": S0,
                "Sigma": BASE_VOLATILITY + bump,
            },
        ]
    )
scenarios = pd.DataFrame(scenario_rows)
scenario_index = {
    name: index for index, name in enumerate(scenarios["Scenario"])
}
display(scenarios)

smoke_uniforms = generate_uniforms("MC", 2**10, N_STEPS, BASE_SEED)
smoke_direct = direct_payoffs(
    smoke_uniforms, [S0], [BASE_VOLATILITY], 0.85 * S0
)
smoke_conditioned = conditioned_payoffs(
    smoke_uniforms, [S0], [BASE_VOLATILITY], 0.85 * S0
)
assert smoke_direct.shape == smoke_conditioned.shape == (2**10, 1)
assert np.all(smoke_direct >= 0) and np.all(smoke_conditioned >= 0)
assert np.isfinite(smoke_conditioned).all()
print("Estimator smoke tests: OK")

## Results

### 3. Construct a high-precision discrete-monitoring reference

The reference uses independent conditioned-RQMC scrambles. The continuous-barrier price is
shown only to quantify the monitoring-convention gap.

In [ ]:
reference_rows = []
reference_start = time.perf_counter()
for replication in range(R_REFERENCE):
    seed = replication_seed("reference", "RQMC", replication)
    uniforms = generate_uniforms(
        "RQMC", N_REFERENCE, N_STEPS, seed
    )
    for barrier_ratio in BARRIER_RATIOS:
        barrier = barrier_ratio * S0
        payoff = conditioned_payoffs(
            uniforms, [S0], [BASE_VOLATILITY], barrier
        )[:, 0]
        reference_rows.append(
            {
                "Barrier ratio": barrier_ratio,
                "Replication": replication,
                "Seed / scramble": seed,
                "Estimate": payoff.mean(),
            }
        )
    if (replication + 1) % 4 == 0:
        print(f"Reference scrambles completed: {replication + 1}/{R_REFERENCE}")

reference_raw = pd.DataFrame(reference_rows)
reference_summary = (
    reference_raw.groupby("Barrier ratio", as_index=False)
    .agg(
        Reference_price=("Estimate", "mean"),
        Reference_replication_SD=("Estimate", "std"),
    )
)
reference_summary["Reference_SE"] = (
    reference_summary["Reference_replication_SD"]
    / np.sqrt(R_REFERENCE)
)
reference_summary["Continuous_analytic"] = [
    continuous_down_and_out_call(
        S0,
        STRIKE,
        barrier_ratio * S0,
        RISK_FREE_RATE,
        DIVIDEND_YIELD,
        BASE_VOLATILITY,
        T,
    )
    for barrier_ratio in reference_summary["Barrier ratio"]
]
reference_summary["Discrete_minus_continuous"] = (
    reference_summary["Reference_price"]
    - reference_summary["Continuous_analytic"]
)
reference_runtime = time.perf_counter() - reference_start
print(f"Reference runtime: {reference_runtime:.1f} seconds")
display(reference_summary)

### 3b. Separate monitoring bias from sampling error

This diagnostic varies the number of monitoring dates and the Sobol sample size
independently for the near-barrier 95% case. Replication SD measures sampling
uncertainty at fixed monitoring, while the bias against the continuous-barrier
analytic value shows the distinct monitoring-discretisation effect. The analytic
value is a limiting diagnostic, not the truth for any finite monitoring grid.

In [ ]:
MONITORING_STEPS = [12, 52, 252]
DIAGNOSTIC_SAMPLE_SIZES = [2**10, 2**12, 2**14]
R_DIAGNOSTIC = 8
DIAGNOSTIC_BARRIER_RATIO = 0.95
diagnostic_rows = []
diagnostic_start = time.perf_counter()

continuous_reference = continuous_down_and_out_call(
    S0,
    STRIKE,
    DIAGNOSTIC_BARRIER_RATIO * S0,
    RISK_FREE_RATE,
    DIVIDEND_YIELD,
    BASE_VOLATILITY,
    T,
)

for monitoring_steps in MONITORING_STEPS:
    for sample_size in DIAGNOSTIC_SAMPLE_SIZES:
        for replication in range(R_DIAGNOSTIC):
            seed = (
                BASE_SEED
                + 200_000_000
                + monitoring_steps * 100_000
                + int(np.log2(sample_size)) * 1_000
                + replication
            )
            uniforms = generate_uniforms(
                "RQMC", sample_size, monitoring_steps, seed
            )
            payoff = conditioned_payoffs(
                uniforms,
                [S0],
                [BASE_VOLATILITY],
                DIAGNOSTIC_BARRIER_RATIO * S0,
            )[:, 0]
            diagnostic_rows.append(
                {
                    "Monitoring steps": monitoring_steps,
                    "Sample size": sample_size,
                    "Replication": replication,
                    "Estimate": payoff.mean(),
                }
            )

monitoring_sample_raw = pd.DataFrame(diagnostic_rows)
monitoring_sample_summary = (
    monitoring_sample_raw.groupby(
        ["Monitoring steps", "Sample size"], as_index=False
    )
    .agg(
        Mean_price=("Estimate", "mean"),
        Replication_SD=("Estimate", "std"),
    )
)
monitoring_sample_summary["Bias_vs_continuous"] = (
    monitoring_sample_summary["Mean_price"] - continuous_reference
)
monitoring_sample_summary["Continuous_analytic"] = continuous_reference
diagnostic_runtime = time.perf_counter() - diagnostic_start

display(monitoring_sample_summary)
print(f"Monitoring/sample-size diagnostic runtime: {diagnostic_runtime:.1f} seconds")
assert (
    monitoring_sample_summary.groupby("Monitoring steps")["Sample size"].nunique()
    == len(DIAGNOSTIC_SAMPLE_SIZES)
).all()
assert (
    monitoring_sample_raw.groupby(["Monitoring steps", "Sample size"]).size()
    == R_DIAGNOSTIC
).all()

### 4. Run the four estimators with CRN across every bump

A single uniform matrix is generated per sampling family and replication. That matrix is reused
across barriers, direct/conditioned treatment, and every spot/volatility bump.

In [ ]:
price_rows = []
delta_rows = []
vega_rows = []
gamma_rows = []
main_start = time.perf_counter()

scenario_spots = scenarios["Spot"].to_numpy()
scenario_sigmas = scenarios["Sigma"].to_numpy()
base_index = scenario_index["base"]

for sampling in ("MC", "RQMC"):
    for replication in range(R_MAIN):
        seed = replication_seed("main", sampling, replication)
        uniforms = generate_uniforms(
            sampling, N_MAIN, N_STEPS, seed
        )
        for barrier_ratio in BARRIER_RATIOS:
            barrier = barrier_ratio * S0

            direct_start = time.perf_counter()
            direct_means = direct_payoffs(
                uniforms,
                scenario_spots,
                scenario_sigmas,
                barrier,
            ).mean(axis=0)
            direct_runtime = time.perf_counter() - direct_start

            conditioned_start = time.perf_counter()
            conditioned_means = conditioned_payoffs(
                uniforms,
                scenario_spots,
                scenario_sigmas,
                barrier,
            ).mean(axis=0)
            conditioned_runtime = (
                time.perf_counter() - conditioned_start
            )

            for treatment, estimates, elapsed in (
                ("Direct", direct_means, direct_runtime),
                ("Conditioned", conditioned_means, conditioned_runtime),
            ):
                method = f"{treatment} {sampling}"
                price_rows.append(
                    {
                        "Method": method,
                        "Sampling": sampling,
                        "Treatment": treatment,
                        "Barrier ratio": barrier_ratio,
                        "Replication": replication,
                        "Seed / scramble": seed,
                        "Price": estimates[base_index],
                        "Runtime seconds": elapsed,
                    }
                )

                for bump in SPOT_BUMPS:
                    down = estimates[
                        scenario_index[f"spot_down_{bump:.4f}"]
                    ]
                    up = estimates[
                        scenario_index[f"spot_up_{bump:.4f}"]
                    ]
                    spot_step = bump * S0
                    delta_rows.append(
                        {
                            "Method": method,
                            "Barrier ratio": barrier_ratio,
                            "Replication": replication,
                            "Spot bump": bump,
                            "Delta": (up - down) / (2.0 * spot_step),
                        }
                    )
                    gamma_rows.append(
                        {
                            "Method": method,
                            "Barrier ratio": barrier_ratio,
                            "Replication": replication,
                            "Spot bump": bump,
                            "Gamma": (
                                up
                                - 2.0 * estimates[base_index]
                                + down
                            )
                            / spot_step**2,
                        }
                    )

                for bump in VOL_BUMPS:
                    down = estimates[
                        scenario_index[f"vol_down_{bump:.4f}"]
                    ]
                    up = estimates[
                        scenario_index[f"vol_up_{bump:.4f}"]
                    ]
                    vega_rows.append(
                        {
                            "Method": method,
                            "Barrier ratio": barrier_ratio,
                            "Replication": replication,
                            "Vol bump": bump,
                            "Vega per vol point": (
                                (up - down) / (2.0 * bump) / 100.0
                            ),
                        }
                    )
        if (replication + 1) % 6 == 0:
            print(
                f"{sampling} replications completed: "
                f"{replication + 1}/{R_MAIN}"
            )

main_runtime = time.perf_counter() - main_start
price_raw = pd.DataFrame(price_rows)
delta_raw = pd.DataFrame(delta_rows)
vega_raw = pd.DataFrame(vega_rows)
gamma_raw = pd.DataFrame(gamma_rows)
print(f"Main experiment runtime: {main_runtime:.1f} seconds")
print(
    f"Rows saved — price {len(price_raw):,}, Delta {len(delta_raw):,}, "
    f"Vega {len(vega_raw):,}, Gamma {len(gamma_raw):,}"
)

assert price_raw.groupby(["Method", "Barrier ratio"]).size().eq(R_MAIN).all()
assert delta_raw.groupby(
    ["Method", "Barrier ratio", "Spot bump"]
).size().eq(R_MAIN).all()
assert vega_raw.groupby(
    ["Method", "Barrier ratio", "Vol bump"]
).size().eq(R_MAIN).all()

### 5. Price variance reduction and estimator agreement

In [ ]:
def root_mean_square(values):
    values = np.asarray(values, dtype=float)
    return float(np.sqrt(np.mean(values**2)))


price_summary = (
    price_raw.groupby(["Method", "Barrier ratio"], as_index=False)
    .agg(
        Mean_price=("Price", "mean"),
        Replication_SD=("Price", "std"),
        Mean_runtime_seconds=("Runtime seconds", "mean"),
    )
    .merge(reference_summary, on="Barrier ratio", how="left")
)
price_summary["Bias_vs_reference"] = (
    price_summary["Mean_price"]
    - price_summary["Reference_price"]
)
rmse_rows = (
    price_raw.merge(
        reference_summary[["Barrier ratio", "Reference_price"]],
        on="Barrier ratio",
    )
    .assign(
        Squared_error=lambda frame: (
            frame["Price"] - frame["Reference_price"]
        )
        ** 2
    )
    .groupby(["Method", "Barrier ratio"], as_index=False)
    .agg(
        RMSE_vs_reference=(
            "Squared_error",
            lambda values: np.sqrt(values.mean()),
        )
    )
)
price_summary = price_summary.merge(
    rmse_rows, on=["Method", "Barrier ratio"]
)
price_summary["SE_of_mean"] = (
    price_summary["Replication_SD"] / np.sqrt(R_MAIN)
)

vrf_rows = []
for barrier_ratio, group in price_raw.groupby("Barrier ratio"):
    variances = group.groupby("Method")["Price"].var().to_dict()
    direct_rqmc_gain = (
        variances["Direct MC"] / variances["Direct RQMC"]
    )
    conditioned_rqmc_gain = (
        variances["Conditioned MC"]
        / variances["Conditioned RQMC"]
    )
    vrf_rows.append(
        {
            "Barrier ratio": barrier_ratio,
            "Conditioning VRF — MC": (
                variances["Direct MC"]
                / variances["Conditioned MC"]
            ),
            "Conditioning VRF — RQMC": (
                variances["Direct RQMC"]
                / variances["Conditioned RQMC"]
            ),
            "RQMC gain — direct": direct_rqmc_gain,
            "RQMC gain — conditioned": conditioned_rqmc_gain,
            "RQMC-after-conditioning interaction": (
                conditioned_rqmc_gain / direct_rqmc_gain
            ),
        }
    )
price_vrf = pd.DataFrame(vrf_rows)

paired = price_raw.pivot_table(
    index=["Sampling", "Barrier ratio", "Replication"],
    columns="Treatment",
    values="Price",
).reset_index()
paired["Direct_minus_conditioned"] = (
    paired["Direct"] - paired["Conditioned"]
)
price_agreement = (
    paired.groupby(["Sampling", "Barrier ratio"], as_index=False)
    .agg(
        Mean_difference=("Direct_minus_conditioned", "mean"),
        SD_of_paired_difference=(
            "Direct_minus_conditioned",
            "std",
        ),
    )
)
price_agreement["SE_of_difference"] = (
    price_agreement["SD_of_paired_difference"] / np.sqrt(R_MAIN)
)
price_agreement["Paired_z"] = (
    price_agreement["Mean_difference"]
    / price_agreement["SE_of_difference"]
)

display(
    price_summary[
        [
            "Method",
            "Barrier ratio",
            "Mean_price",
            "Reference_price",
            "Bias_vs_reference",
            "Replication_SD",
            "RMSE_vs_reference",
        ]
    ]
)
display(price_vrf)
display(price_agreement)
print(
    "Maximum absolute paired direct-conditioned z-score:",
    f"{price_agreement['Paired_z'].abs().max():.2f}",
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

for method in METHOD_ORDER:
    subset = price_summary[price_summary["Method"] == method]
    axes[0].errorbar(
        100 * subset["Barrier ratio"],
        subset["Mean_price"],
        yerr=subset["Replication_SD"],
        marker="o",
        capsize=3,
        color=METHOD_COLORS[method],
        label=method,
    )
axes[0].plot(
    100 * reference_summary["Barrier ratio"],
    reference_summary["Reference_price"],
    color="black",
    linestyle="--",
    marker="x",
    label="Conditioned-RQMC reference",
)
axes[0].set(
    xlabel="Barrier / initial spot (%)",
    ylabel="Price per 100 notional",
    title="Price mean ± replication SD",
)
axes[0].legend(fontsize=8)

x = np.arange(len(price_vrf))
width = 0.36
axes[1].bar(
    x - width / 2,
    price_vrf["Conditioning VRF — MC"],
    width,
    label="Conditioning VRF — MC",
    color="#54A24B",
)
axes[1].bar(
    x + width / 2,
    price_vrf["Conditioning VRF — RQMC"],
    width,
    label="Conditioning VRF — RQMC",
    color="#E45756",
)
axes[1].axhline(1.0, color="black", linewidth=1, linestyle="--")
axes[1].set(
    xticks=x,
    xticklabels=[
        f"{100 * value:.0f}%"
        for value in price_vrf["Barrier ratio"]
    ],
    xlabel="Barrier / initial spot",
    ylabel="Variance-reduction factor",
    title="Direct variance / conditioned variance",
)
axes[1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "price_variance_reduction.png", dpi=180)
plt.show()

### 6. Delta dispersion across the full spot-bump grid

In [ ]:
def median_absolute_deviation(values):
    values = np.asarray(values, dtype=float)
    median = np.median(values)
    return float(np.median(np.abs(values - median)))


delta_summary = (
    delta_raw.groupby(
        ["Method", "Barrier ratio", "Spot bump"],
        as_index=False,
    )
    .agg(
        Mean_Delta=("Delta", "mean"),
        Replication_SD=("Delta", "std"),
        MAD=("Delta", median_absolute_deviation),
        P05=("Delta", lambda values: np.quantile(values, 0.05)),
        P95=("Delta", lambda values: np.quantile(values, 0.95)),
    )
)
display(delta_summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
for axis, barrier_ratio in zip(axes, BARRIER_RATIOS):
    subset = delta_summary[
        delta_summary["Barrier ratio"] == barrier_ratio
    ]
    for method in METHOD_ORDER:
        method_data = subset[subset["Method"] == method]
        axis.plot(
            100 * method_data["Spot bump"],
            method_data["Replication_SD"],
            marker="o",
            color=METHOD_COLORS[method],
            label=method,
        )
    axis.set(
        title=f"Barrier {100 * barrier_ratio:.0f}%",
        xlabel="Spot bump (%)",
        ylabel="Delta replication SD",
    )
axes[-1].legend(fontsize=8)
plt.suptitle("Delta dispersion under CRN")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "delta_dispersion.png", dpi=180)
plt.show()

### 7. Vega dispersion across three volatility bumps

In [ ]:
vega_summary = (
    vega_raw.groupby(
        ["Method", "Barrier ratio", "Vol bump"],
        as_index=False,
    )
    .agg(
        Mean_Vega_per_vol_point=(
            "Vega per vol point",
            "mean",
        ),
        Replication_SD=("Vega per vol point", "std"),
        MAD=("Vega per vol point", median_absolute_deviation),
    )
)
display(vega_summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
for axis, barrier_ratio in zip(axes, BARRIER_RATIOS):
    subset = vega_summary[
        vega_summary["Barrier ratio"] == barrier_ratio
    ]
    for method in METHOD_ORDER:
        method_data = subset[subset["Method"] == method]
        axis.plot(
            100 * method_data["Vol bump"],
            method_data["Replication_SD"],
            marker="o",
            color=METHOD_COLORS[method],
            label=method,
        )
    axis.set(
        title=f"Barrier {100 * barrier_ratio:.0f}%",
        xlabel="Volatility bump (vol points)",
        ylabel="Vega replication SD",
    )
axes[-1].legend(fontsize=8)
plt.suptitle("Vega dispersion under CRN")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "vega_dispersion.png", dpi=180)
plt.show()

### 8. Gamma versus spot bump, with the 95% near-barrier case highlighted

Gamma uses a central second difference. Instability is evaluated through multiple CRN bumps and
the full replication distribution; a jagged mean curve alone is not treated as evidence.

In [ ]:
gamma_summary = (
    gamma_raw.groupby(
        ["Method", "Barrier ratio", "Spot bump"],
        as_index=False,
    )
    .agg(
        Mean_Gamma=("Gamma", "mean"),
        Replication_SD=("Gamma", "std"),
        MAD=("Gamma", median_absolute_deviation),
        P05=("Gamma", lambda values: np.quantile(values, 0.05)),
        P95=("Gamma", lambda values: np.quantile(values, 0.95)),
    )
)
near_gamma = gamma_summary[
    np.isclose(gamma_summary["Barrier ratio"], 0.95)
].copy()
gamma_stability = (
    near_gamma.groupby("Method", as_index=False)
    .agg(
        Min_mean_Gamma=("Mean_Gamma", "min"),
        Max_mean_Gamma=("Mean_Gamma", "max"),
        Mean_replication_SD=("Replication_SD", "mean"),
        Max_replication_SD=("Replication_SD", "max"),
    )
)
gamma_stability["Cross_bump_range"] = (
    gamma_stability["Max_mean_Gamma"]
    - gamma_stability["Min_mean_Gamma"]
)
display(near_gamma)
display(gamma_stability)

fig, axis = plt.subplots(figsize=(8.8, 5.2))
for method in METHOD_ORDER:
    method_data = near_gamma[near_gamma["Method"] == method]
    axis.errorbar(
        100 * method_data["Spot bump"],
        method_data["Mean_Gamma"],
        yerr=method_data["Replication_SD"],
        marker="o",
        capsize=3,
        color=METHOD_COLORS[method],
        label=method,
    )
axis.axhline(0.0, color="black", linewidth=1)
axis.set(
    xlabel="Spot bump (%)",
    ylabel="Gamma (mean ± replication SD)",
    title="Near-barrier Gamma stability — barrier at 95% of initial spot",
)
axis.legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "gamma_vs_spot_bump_95pct.png", dpi=180)
plt.show()

### 9. Does RQMC improve more after conditioning?

An interaction ratio above one means the MC-to-RQMC variance gain is larger for the conditioned
estimator than for the direct estimator.

In [ ]:
interaction_table = price_vrf[
    [
        "Barrier ratio",
        "RQMC gain — direct",
        "RQMC gain — conditioned",
        "RQMC-after-conditioning interaction",
    ]
].copy()
interaction_table["RQMC improves more after conditioning?"] = np.where(
    interaction_table["RQMC-after-conditioning interaction"] > 1.0,
    "Yes",
    "No",
)
display(interaction_table)

fig, axis = plt.subplots(figsize=(7.5, 4.4))
axis.plot(
    100 * interaction_table["Barrier ratio"],
    interaction_table["RQMC gain — direct"],
    marker="o",
    label="RQMC gain — direct",
)
axis.plot(
    100 * interaction_table["Barrier ratio"],
    interaction_table["RQMC gain — conditioned"],
    marker="s",
    label="RQMC gain — conditioned",
)
axis.set_yscale("log")
axis.set(
    xlabel="Barrier / initial spot (%)",
    ylabel="MC variance / RQMC variance (log scale)",
    title="RQMC variance gain before and after conditioning",
)
axis.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "rqmc_conditioning_interaction.png", dpi=180)
plt.show()

### 10. Save the auditable replication outputs

In [ ]:
reference_raw.to_csv(OUTPUT_DIR / "reference_replications.csv", index=False)
reference_summary.to_csv(OUTPUT_DIR / "reference_summary.csv", index=False)
monitoring_sample_raw.to_csv(OUTPUT_DIR / "monitoring_sample_replications.csv", index=False)
monitoring_sample_summary.to_csv(OUTPUT_DIR / "monitoring_sample_grid.csv", index=False)
price_raw.to_csv(OUTPUT_DIR / "price_replications.csv", index=False)
price_summary.to_csv(OUTPUT_DIR / "price_summary.csv", index=False)
price_vrf.to_csv(OUTPUT_DIR / "price_variance_reduction.csv", index=False)
price_agreement.to_csv(OUTPUT_DIR / "price_method_agreement.csv", index=False)
delta_raw.to_csv(OUTPUT_DIR / "delta_replications.csv", index=False)
delta_summary.to_csv(OUTPUT_DIR / "delta_summary.csv", index=False)
vega_raw.to_csv(OUTPUT_DIR / "vega_replications.csv", index=False)
vega_summary.to_csv(OUTPUT_DIR / "vega_summary.csv", index=False)
gamma_raw.to_csv(OUTPUT_DIR / "gamma_replications.csv", index=False)
gamma_summary.to_csv(OUTPUT_DIR / "gamma_summary.csv", index=False)
gamma_stability.to_csv(OUTPUT_DIR / "gamma_stability_95pct.csv", index=False)
interaction_table.to_csv(OUTPUT_DIR / "rqmc_interaction.csv", index=False)

run_manifest = pd.DataFrame(
    [
        {
            "Project-relative source": SOURCE_FILE.relative_to(PROJECT_DIR).as_posix(),
            "Source SHA-256": SOURCE_SHA256,
            "Frozen config": CONFIG_FILE.relative_to(PROJECT_DIR).as_posix(),
            "Underlying": market["ticker"],
            "Model": "Risk-neutral GBM",
            "Product": "Monthly monitored down-and-out call",
            "S0": S0,
            "Strike": STRIKE,
            "Maturity": T,
            "Monitoring steps": N_STEPS,
            "Barrier ratios": ", ".join(map(str, BARRIER_RATIOS)),
            "Main sample size": N_MAIN,
            "Main replications": R_MAIN,
            "Reference sample size": N_REFERENCE,
            "Reference scrambles": R_REFERENCE,
            "Diagnostic monitoring steps": ", ".join(map(str, MONITORING_STEPS)),
            "Diagnostic sample sizes": ", ".join(map(str, DIAGNOSTIC_SAMPLE_SIZES)),
            "Diagnostic scrambles": R_DIAGNOSTIC,
            "Spot bumps": ", ".join(map(str, SPOT_BUMPS)),
            "Volatility bumps": ", ".join(map(str, VOL_BUMPS)),
            "CRN": "Same uniforms across treatment, barriers and bumps within sampling replication",
            "RQMC": "Independent Owen-scrambled Sobol replications",
            "Uniform clipping epsilon": UNIFORM_EPS,
            "Base seed": BASE_SEED,
            "Main runtime seconds": main_runtime,
            "Reference runtime seconds": reference_runtime,
            "Diagnostic runtime seconds": diagnostic_runtime,
        }
    ]
)
run_manifest.to_csv(OUTPUT_DIR / "run_manifest.csv", index=False)

print(f"Saved outputs to: {OUTPUT_DIR}")
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))

## Takeaways

1. **Gate 2 passes.** Direct and one-step survival-conditioned prices are statistically
   consistent for all three barrier distances and both sampling families; the maximum
   absolute paired \(z\)-score is 1.43.

2. **Conditioning is regime-dependent rather than universally dominant.** It does
   almost nothing at the 70% barrier, becomes useful at 85%, and is decisive at 95%.
   This is the expected economic pattern: smoothing matters when barrier crossings are
   a material part of the payoff distribution.

3. **RQMC and conditioning are complementary.** The RQMC variance gain is only
   marginally larger after conditioning at 70%, but the interaction ratio rises to
   2.49× at 85% and 13.48× at 95%. Smoothing the indicator makes the Sobol construction
   much more effective.

4. **CRN does not rescue a discontinuous Gamma by itself.** Even though every spot bump
   shares the same uniforms, direct Gamma remains highly sensitive to bump size and
   replication. That instability is therefore a substantive estimator result, not a
   seed artefact.

5. **Conditioning stabilises all three Greeks.** Delta and Vega dispersion become almost
   flat across the requested bump grids, while conditioned Gamma exhibits a clear
   plateau near 0.0144 in the 95% case. Conditioned RQMC is the most stable of the four
   methods in this experiment.

6. **Monitoring convention matters.** The numerical reference prices the monthly
   discrete product. The continuous-barrier analytic value is lower, especially at the
   95% barrier, and should not be used as the “truth” for the discrete estimator.


7. **Sampling error and monitoring bias are reported separately.** The monitoring/sample-size grid changes Sobol sample size within each monthly, weekly and daily monitoring convention, so convergence in \(N\) is not confused with convergence of the monitoring grid.